# 14 · CP5 evidence faithfulness and the P6 explanation pass (L4)

**L4 runtime**, one session (Dhanush, 19 Nov 2026). About **2.5–3 compute units** at the measured 2.2 CU/h (estimate).

**Model:** the unified model, final configuration (context crops), seed 42: `runs/eviground_ctx_unified_seed42/best.pt`.

**Part 1 · CP5 go/no-go, validation images only** (spec §9.1; training editors, no test image):
- **EPG** on 200 edited validation images. Pass: mean EPG ≥ 1.5 × the mean area fraction of the edit **and** a one-sided paired Wilcoxon test with p < 0.01.
- **Randomization sanity test** on 30 of them (heads → fusion → trace branch → LoRA re-initialized, one step after another). Pass: mean SSIM between the trained and the fully randomized map < 0.5.
- **Deletion** on the tile with the largest edited share. Pass: mean deletion AUC below the random-order deletion AUC.
- **20 evidence overlays**: 10 edited, 5 generated, 5 real.
- **Both maps are checked:** Grad-CAM++ and the mask decoder.
  - Grad-CAM++ is the primary evidence map if it passes all three (`PASS`).
  - Otherwise the mask decoder is primary and Grad-CAM++ secondary (`FALLBACK`, the spec's fix).
  - If both fail: `FAIL`, stop and ask Dhanush.

**Part 2 · P6** (frozen by notebook 12b: 1,500 images):
- an evidence bundle per image (evidence JSON, region masks, maps, probabilities, gate) for CP6;
- EPG for both maps on the 1,200 edited images;
- deletion / insertion on `P6_deletion` (300);
- randomization on 50 edited images.
- Both maps are saved; the CP5 verdict says which one is primary.
- **A dropped session resumes:** images already written are skipped.

**Results** (send to Claude):
- `Paper 1/runs/cp5/cp5_evidence_faithfulness_seed42.json`;
- `p6_faithfulness_seed42.csv`;
- `cp5_overlays_val.png`;
- `compute_cp5.json`.
- The bundles are in `Paper 1/runs/explain/p6_unified_ctx_seed42/`.

In [ ]:
# Cell 1: mount Drive, get the code, install it
import os, subprocess, sys

DRY = os.environ.get('EVIGROUND_DRY_RUN') == '1'   # CI / local test: fake sources, temp folders
REPO_URL = 'github.com/DHANUSHKUMMARI/Paper1.git'
BRANCH = 'main'                                    # 'claude/nice-curie-3tzxzr' before the merge

if DRY:
    REPO = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
else:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO = '/content/Paper1'
    if not os.path.exists(REPO):
        try:
            tok = userdata.get('GITHUB_TOKEN')       # never printed
        except Exception:
            tok = None
        url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
        r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO],
                           capture_output=True, text=True)
        if r.returncode != 0:                        # stderr is not shown: it can contain the URL
            raise RuntimeError('git clone failed: check GITHUB_TOKEN (Secrets panel) and BRANCH')
        subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'],
                       check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO + '[train]'], check=True)
    # Colab ships torchao 0.10; peft refuses a torchao older than 0.16 with an ImportError when it
    # adds the LoRA layers, although EviGround never uses torchao: remove an old one.
    import importlib, importlib.metadata as _md
    from packaging.version import Version
    try:
        if Version(_md.version('torchao')) < Version('0.16.0'):
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=True)
            importlib.invalidate_caches()
    except _md.PackageNotFoundError:
        pass

sys.path.insert(0, os.path.join(REPO, 'src'))
# Re-running in a runtime that already imported eviground would keep the OLD code in memory
# although git pulled the new one: drop it so the next import reads the files on disk.
for _name in [m for m in sys.modules if m == 'eviground' or m.startswith('eviground.')]:
    del sys.modules[_name]
print('repo:', REPO, '| branch:', BRANCH, '| dry run:', DRY)

In [ ]:
# Cell 2: settings and pre-flight
import csv, datetime, json, logging, shutil, subprocess, tempfile, time, zipfile
from collections import Counter
from pathlib import Path
logging.basicConfig(level=logging.INFO, force=True)
import numpy as np, torch
from eviground.config import load_config
from eviground.data import brgen, chameleon, pack
from eviground.data.cp1 import _read_csv_gz
from eviground.env import get_paths, session_info
from eviground.eval import p6data, runner, subsets, testsets
from eviground.explain import cp5, evidence, faithfulness
from eviground.utils import session

SEED = 42
CU_PER_HOUR = 2.2          # the measured L4 rate (18 Nov 2026): Colab's balance fell ~2.2 CU per hour
CU_LEFT = None             # optional: the balance shown in Colab; the pre-flight stops if EST_CU exceeds it
EST_CU = 3.0               # estimate ~2.5 (about 1.1 h); 3.0 for margin
EXPERIMENT = 'eviground_ctx'
N_VAL_EDITED, N_VAL_RAND, N_P6_RAND = 200, 30, 50
N_OVERLAY = {'edited': 10, 'generated': 5, 'real': 5}
STEPS = faithfulness.STEPS  # deletion / insertion steps
INDEX = 'index.parquet'    # the shard index
T0 = time.time()
info = session_info(cu_per_hour=CU_PER_HOUR)
paths = get_paths()
if DRY:
    from eviground.env import Paths
    _dry = Path(tempfile.gettempdir()) / 'eviground_cp5_dry'
    shutil.rmtree(_dry, ignore_errors=True)
    paths = Paths(root=_dry / 'drive' / 'Paper 1', local_data=_dry / 'data')
    N_VAL_EDITED, N_VAL_RAND, N_P6_RAND, STEPS, EST_CU = 3, 2, 2, 4, 0.0
    N_OVERLAY = {'edited': 2, 'generated': 1, 'real': 1}
paths.ensure()
CONFIG = 'debug' if DRY else 'base'
HF_CACHE = paths.cache / 'hf' if DRY else Path('/content/hf_cache')
OVERRIDES = [f'model.semantic.cache_dir={HF_CACHE}', f'compute.cu_per_hour={CU_PER_HOUR}']
cfg = load_config(CONFIG, OVERRIDES)
if not DRY:
    assert torch.cuda.is_available(), 'needs a GPU runtime (L4)'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = lambda: runner.autocast(DEVICE)  # noqa: E731  (bf16 on the L4)
CK = paths.runs / 'checkpoints'
CP1_MANIFEST = CK / 'cp1' / 'manifest_cp1.csv.gz'
SUBSETS_FILE = CK / 'eval_subsets.json'
RUN = paths.runs / f'{EXPERIMENT}_unified_seed{SEED}'
BEST = RUN / 'best.pt'
OUT = paths.runs / 'cp5'
BUNDLES = paths.runs / 'explain' / f'p6_unified_ctx_seed{SEED}'
JSONL = BUNDLES / 'p6_records.jsonl'
BRGEN_ARCHIVE = paths.datasets / '04_BR-Gen' / brgen.LAST_VOLUME
CHAM_ZIP = paths.datasets / '05_Chameleon' / chameleon.ZIP_NAME
RESULT = OUT / f'cp5_evidence_faithfulness_seed{SEED}.json'
OUT.mkdir(parents=True, exist_ok=True); BUNDLES.mkdir(parents=True, exist_ok=True)
print('commit', info['git_commit'], '| gpu', info.get('gpu_name'), '| device', DEVICE, '| model', BEST)

In [ ]:
# Cell 3: shards (validation images) and, in the dry run, the fake world
SHARDS = paths.shards
if DRY:   # fake world -> CP1 -> shards -> a tiny unified model (one epoch); a small fake P6
    from eviground.data.cp1 import CP1Config, run_cp1
    from eviground.data.download import ensure_data
    from eviground.data.fake_cp1 import cp1_overrides, make_cp1_world
    from eviground.train.trainer import Trainer
    sources = make_cp1_world(paths.datasets, paths.root.parent / 'fake_remote', coco_images=40)
    ensure_data('cp1', paths=paths, sources=sources, stage_local=True, overrides=cp1_overrides(40))
    run_cp1(paths, CP1Config(workers=2, crops_per_class=60, blockiness_per_group=20, base_unclear_max=0.2,
                             diffseg_card_splits={'train': 18, 'validation': 12}),
            out_dir=CK / 'cp1', cache_dir=paths.cache / 'cp1', progress=None)
    pack.pack(CP1_MANIFEST, SHARDS, free_bytes=10**12, margin=0, shard_bytes=200_000, sample_per_source=5,
              progress=None)
    Trainer(load_config(CONFIG, OVERRIDES + ['train.epochs=1', 'data.context_crop_share=0.3333']), SHARDS, RUN,
            seed=SEED, profile='unified', resume=None, log=None).fit()
    import hashlib
    rows = _read_csv_gz(CP1_MANIFEST)
    pick = lambda f, n: sorted((r for r in rows if f(r)), key=lambda r: r['uid'])[:n]  # noqa: E731
    parts = {'diffseg_heldout': pick(lambda r: r['role'] == 'test_P2', 2),
             'sidset_tampered': pick(lambda r: r['role'] == 'test_P3b' and r['label'] == 'edited' and r['mask_path'], 2),
             'generated': pick(lambda r: r['role'] == 'test_P1' and r['label'] == 'generated', 1),
             'real': pick(lambda r: r['role'] == 'test_P1' and r['label'] == 'real', 1)}
    from PIL import Image as _PILImage
    for part in ('sidset_tampered', 'diffseg_heldout'):   # half-size masks, as some real P6 masks are (24 Nov)
        mp = Path(parts[part][0]['mask_path'])
        with _PILImage.open(mp) as _m:
            _m.resize((max(1, _m.width // 2), max(1, _m.height // 2)), _PILImage.NEAREST).save(mp)
    rgb = np.full((300, 280, 3), 90, np.uint8); m = np.zeros((300, 280), np.uint8); m[40:160, 60:200] = 255
    BRGEN_ARCHIVE.parent.mkdir(parents=True, exist_ok=True); CHAM_ZIP.parent.mkdir(parents=True, exist_ok=True)
    br_files = {}
    with zipfile.ZipFile(BRGEN_ARCHIVE, 'w') as zf:
        for i in range(2):
            f, mk = f'BR-Gen/Forged/LaMa/Stuff/COCO/{i}_stuff.png', f'BR-Gen/Mask/Stuff/COCO/{i}_stuff.png'
            fb, mb = p6data.png_bytes(rgb + i), p6data.png_bytes(m)
            zf.writestr(f, fb); zf.writestr(mk, mb)
            br_files[f] = {'sha256': hashlib.sha256(fb).hexdigest(), 'mask': mk, 'mask_sha256': hashlib.sha256(mb).hexdigest()}
    cham = 'Chameleon/test/0_real/r0.png'
    with zipfile.ZipFile(CHAM_ZIP, 'w') as zf:
        zf.writestr(cham, p6data.png_bytes(rgb))
    part_of = {r['uid']: k for k, rs in parts.items() for r in rs}
    part_of.update(dict.fromkeys(br_files, 'brgen')); part_of[cham] = 'real'
    chosen = [{'uid': u} for u in part_of]
    edited_uids = [u for u, k in part_of.items() if k in p6data.EDITED_PARTS]
    P6_FAKE = subsets._record('P6', chosen, SEED, 'dry', part_of=part_of)
    subsets.save({'P6': P6_FAKE, 'P6_mllm': subsets._record('P6_mllm', chosen, SEED, 'dry'),
                  'P6_deletion': subsets._record('P6_deletion', [{'uid': u} for u in edited_uids[:3]], SEED, 'dry'),
                  'P6_brgen_files': {'files': br_files}}, SUBSETS_FILE)
PREFLIGHT = session.preflight(
    {'notebook': '14 CP5 + P6 explanations', 'experiment': EXPERIMENT, 'model': BEST, 'result file': RESULT,
     'bundles': BUNDLES, 'validation': f'{N_VAL_EDITED} edited, randomization {N_VAL_RAND}',
     'P6': f'all images; deletion on P6_deletion; randomization {N_P6_RAND}'},
    paths={'unified checkpoint (final configuration, seed 42)': BEST, 'shards': SHARDS / INDEX,
           'CP1 manifest': CP1_MANIFEST, 'eval_subsets.json (P6)': SUBSETS_FILE,
           'BR-Gen archive': BRGEN_ARCHIVE, 'Chameleon.zip': CHAM_ZIP},
    expect={'experiment': 'eviground_ctx'}, est_cu=EST_CU, cu_left=CU_LEFT)
STORED = json.loads(SUBSETS_FILE.read_text())
for k in ('P6', 'P6_mllm', 'P6_deletion'):
    assert subsets.intact(STORED.get(k, {})), f'{SUBSETS_FILE}: {k} is missing or was changed: stop and ask Dhanush'
P6 = STORED['P6']
LOCAL_SHARDS = paths.local_data / 'shards'
print(pack.stage_shards(SHARDS, LOCAL_SHARDS, progress=print))
SHARDS = LOCAL_SHARDS

In [ ]:
# Cell 4: the P6 files on the local disk (DiffSeg30k and SID-Set staged copies, GenImage copies,
# the Chameleon and BR-Gen members P6 needs), with the BR-Gen hash check of notebook 12b
t = time.time()
MANIFEST = _read_csv_gz(CP1_MANIFEST)
ROWS = p6data.manifest_rows(P6, MANIFEST)
if not DRY:
    from eviground.data.download import ensure_data
    ensure_data(datasets=['diffseg30k', 'sidset'], paths=paths, stage_local=True)
gi = [r for r in ROWS.values() if r.get('source') == 'genimage']
for r in testsets.stage_files(gi, paths.local_data / 'p6_genimage'):
    ROWS[r['uid']] = r
CHAM_DIR = paths.local_data / 'p6_chameleon'
cham_uids = [u for u in P6['uids'] if u not in ROWS and P6['part_of'][u] == 'real']
with zipfile.ZipFile(CHAM_ZIP) as zf:
    for u in cham_uids:
        zf.extract(u, CHAM_DIR)
BR_FILES = STORED['P6_brgen_files']['files']
BR_DIR = paths.local_data / 'p6_brgen'
br_members = [u for u in P6['uids'] if P6['part_of'][u] == 'brgen']
exe = brgen.seven_zip()
if exe is None and not DRY:
    subprocess.run(['apt-get', 'install', '-y', '-q', 'p7zip-full'], check=True, capture_output=True)
    exe = brgen.seven_zip()
assert exe, '7z is not available'
brgen.extract(BRGEN_ARCHIVE, br_members + sorted({BR_FILES[u]['mask'] for u in br_members}), BR_DIR, exe)
BAD = p6data.check_brgen({u: BR_FILES[u] for u in br_members}, BR_DIR)
if BAD:
    raise RuntimeError(f'BR-Gen files differ from notebook 12b: {BAD[:5]}: stop and ask Dhanush')
STAGE_MIN = round((time.time() - t) / 60, 2)
print(f"P6: {len(ROWS)} manifest images, {len(cham_uids)} Chameleon, {len(br_members)} BR-Gen (hashes checked) "
      f"| staged in {STAGE_MIN} min")

In [ ]:
# Cell 5: the model
model = runner.load_trained(cfg.model, BEST, DEVICE)
print('model loaded:', BEST, '| edit threshold', model.edit_threshold)

In [ ]:
# Cell 6: CP5 on validation images (training editors; no test image)
from eviground.data.pack import read_index
from eviground.data import tiles as tiling
val = [r for r in read_index(SHARDS / INDEX) if r['split'] == 'val']
by = {lab: [r for r in val if r['label'] == lab] for lab in ('edited', 'generated', 'real')}
if DRY and not by['edited']:   # the fake world's validation split has no edited image (as in notebook 08's dry run)
    by['edited'] = [r for r in read_index(SHARDS / INDEX) if r['split'] == 'train' and r['label'] == 'edited']
assert by['edited'], 'no edited validation image in the shards: stop and ask Dhanush'
V_EDITED = subsets.take(by['edited'], N_VAL_EDITED, SEED, 'cp5_val_v1/edited')
V_OTHER = (subsets.take(by['generated'], N_OVERLAY['generated'], SEED, 'cp5_val_v1/generated')
           + subsets.take(by['real'], N_OVERLAY['real'], SEED, 'cp5_val_v1/real'))
overlay_uids = {r['uid'] for r in V_EDITED[:N_OVERLAY['edited']]} | {r['uid'] for r in V_OTHER}
VAL_RECORDS, PANELS, RAND_TILES = [], [], []
t = time.time()
for item in testsets.shard_items(SHARDS, V_EDITED + V_OTHER):
    out = cp5.explain_image(model, item.image, DEVICE, AMP)
    maps = {'gradcam': out['gradcam'], 'decoder': out['decoder']}
    if item.mask is not None and item.mask.any():
        rec = cp5.faithfulness_record(model, item.image, item.mask, maps, DEVICE, steps=STEPS, seed=SEED, amp=AMP)
        VAL_RECORDS.append({'uid': item.uid, 'source': item.meta.get('source'), 'verdict': out['verdict'], **rec})
        if len(RAND_TILES) < N_VAL_RAND:
            h, w = item.mask.shape
            le, tp, ri, bo = faithfulness.best_tile(item.mask, tiling.tile_boxes(h, w))
            RAND_TILES.append(cp5.to_tensor(item.image, DEVICE)[:, tp:bo, le:ri])
    if item.uid in overlay_uids:
        record, _ = evidence.evidence_record(out['probs'], out['gate'], out['decoder'], out['gradcam_pred'], item.image)
        PANELS.append({'image': item.image, 'mask': item.mask, 'record': record,
                       'maps': {'decoder': out['decoder'], 'gradcam_pred': out['gradcam_pred']},
                       'title': f"{item.meta.get('label')} -> {out['verdict']} "
                                f"(P edited {out['probs']['edited']:.2f}, g {out['gate']:.2f})"})
VAL_RAND = cp5.randomization(model, torch.stack(RAND_TILES), seed=SEED, amp=AMP)
VAL_VERDICT = cp5.cp5_verdict(VAL_RECORDS, VAL_RAND)
VAL_MIN = round((time.time() - t) / 60, 2)
print(f'CP5 on {len(VAL_RECORDS)} edited validation images ({VAL_MIN} min): {VAL_VERDICT["verdict"]}, '
      f'primary map {VAL_VERDICT["primary_map"]}')
for name, v in VAL_VERDICT['maps'].items():
    print(f"  {name:8} EPG {v['epg']['mean_epg']:.3f} vs area {v['epg']['mean_area_fraction']:.3f} "
          f"(x{v['epg']['ratio']:.2f}, p {v['epg']['wilcoxon_p']:.2g}) | SSIM last {v['randomization']['ssim_last']:.3f} "
          f"| deletion {v['deletion']['mean_auc']:.3f} vs random {v['deletion']['mean_random_auc']:.3f} | pass {v['pass']}")
if VAL_VERDICT['verdict'] == 'FAIL':
    print('CP5 FAIL: neither map passes. The P6 pass still runs (both maps are saved); ask Dhanush before using them.')

In [ ]:
# Cell 7: P6, every image: evidence bundle (for CP6) and faithfulness on edited images. Resumes.
done = set()
if JSONL.is_file():
    done = {json.loads(line)['uid'] for line in JSONL.read_text().splitlines() if line.strip()}
DELETION = set(STORED['P6_deletion']['uids'])
todo = [u for u in P6['uids'] if u not in done]
print(f'P6: {len(done)} images done earlier, {len(todo)} to go')
t = time.time()
brgen_masks = {u: BR_FILES[u]['mask'] for u in BR_FILES}
items = p6data.p6_items(todo, P6['part_of'], ROWS, brgen_dir=BR_DIR, brgen_masks=brgen_masks, chameleon_dir=CHAM_DIR)
with open(JSONL, 'a') as fh:
    for n, item in enumerate(items, 1):
        out = cp5.explain_image(model, item.image, DEVICE, AMP)
        record, rles = evidence.evidence_record(out['probs'], out['gate'], out['decoder'], out['gradcam_pred'], item.image)
        evidence.save_bundle(BUNDLES, item.uid, record, rles,
                             {'decoder': out['decoder'], 'gradcam': out['gradcam'], 'gradcam_pred': out['gradcam_pred']},
                             out['probs'], out['gate'])
        line = {'uid': item.uid, 'part': item.meta['part'], 'label': item.meta['label'], 'verdict': out['verdict'],
                'tiles': out['tiles'], 'gate': out['gate'], 'regions': len(record['regions']),
                **{f'p_{k}': v for k, v in out['probs'].items()}}
        if item.mask is not None and item.mask.any():
            line.update(cp5.faithfulness_record(model, item.image, item.mask,
                                                {'gradcam': out['gradcam'], 'decoder': out['decoder']}, DEVICE,
                                                deletion=item.uid in DELETION, steps=STEPS, seed=SEED, amp=AMP))
        fh.write(json.dumps(line) + '\n'); fh.flush()
        if n % 100 == 0:
            print(f'  {n}/{len(todo)} images, {(time.time() - t) / 60:.1f} min')
P6_MIN = round((time.time() - t) / 60, 2)
P6_RECORDS = [json.loads(line) for line in JSONL.read_text().splitlines() if line.strip()]
assert len({r['uid'] for r in P6_RECORDS}) == P6['images'], 'P6 records are incomplete'
print(f'P6 pass: {len(P6_RECORDS)} images ({P6_MIN} min this session)')

In [ ]:
# Cell 8: randomization on P6 edited images (best tiles), the P6 summary and the result file
from eviground.data import tiles as tiling
RAND_P6_FILE = OUT / f'p6_randomization_seed{SEED}.json'
if RAND_P6_FILE.is_file():
    P6_RAND = json.loads(RAND_P6_FILE.read_text())
else:
    pick = sorted(DELETION)[:N_P6_RAND]
    tl = []
    for item in p6data.p6_items(pick, P6['part_of'], ROWS, brgen_dir=BR_DIR, brgen_masks=brgen_masks, chameleon_dir=CHAM_DIR):
        h, w = item.mask.shape
        le, tp, ri, bo = faithfulness.best_tile(item.mask, tiling.tile_boxes(h, w))
        tl.append(cp5.to_tensor(item.image, DEVICE)[:, tp:bo, le:ri])
    P6_RAND = cp5.randomization(model, torch.stack(tl), seed=SEED, amp=AMP)
    RAND_P6_FILE.write_text(json.dumps(P6_RAND))
EDITED_P6 = [r for r in P6_RECORDS if 'epg_gradcam' in r]
P6_MAPS = {part: {name: cp5.map_verdict([r for r in EDITED_P6 if part == 'all' or r['part'] == part], P6_RAND[name], name)
                  for name in cp5.MAPS}
           for part in ('all', *p6data.EDITED_PARTS)}
RESULT_DATA = {
    'step': 'CP5 evidence faithfulness + P6 explanation pass', 'dry_run': DRY, 'branch': BRANCH, 'seed': SEED,
    'experiment': EXPERIMENT, 'model': str(BEST), 'session': info,
    'cp5_validation': {**VAL_VERDICT, 'edited_images': len(VAL_RECORDS), 'randomization_tiles': len(RAND_TILES),
                       'minutes': VAL_MIN},
    'p6': {'sha256': P6['sha256'], 'images': len(P6_RECORDS), 'edited_images': len(EDITED_P6),
           'deletion_images': sum('deletion_gradcam' in r for r in EDITED_P6),
           'faithfulness_by_part': P6_MAPS, 'summary_by_part': cp5.summarize(P6_RECORDS),
           'verdicts_by_label': {lab: dict(Counter(r['verdict'] for r in P6_RECORDS if r['label'] == lab))
                                 for lab in sorted({r['label'] for r in P6_RECORDS})},
           'note': 'P6 numbers are reported for the paper; the CP5 go/no-go is the validation verdict only'},
    'staging_minutes': STAGE_MIN, 'p6_minutes_this_session': P6_MIN,
    'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
}
RESULT.write_text(json.dumps(RESULT_DATA, indent=1, default=str))
with open(OUT / f'p6_faithfulness_seed{SEED}.csv', 'w', newline='') as fh:
    cols = sorted({k for r in P6_RECORDS for k in r})
    w = csv.DictWriter(fh, fieldnames=cols); w.writeheader(); w.writerows(P6_RECORDS)
print('saved', RESULT)
for name in cp5.MAPS:
    a = P6_MAPS['all'][name]
    d = a['deletion']
    dele = 'n/a' if d['mean_auc'] is None else f"{d['mean_auc']:.3f} vs random {d['mean_random_auc']:.3f}"
    print(f"P6 {name:8} EPG {a['epg']['mean_epg']:.3f} vs area {a['epg']['mean_area_fraction']:.3f} | "
          f"deletion {dele} | SSIM last {a['randomization']['ssim_last']:.3f}")
if DRY:
    assert len(P6_RECORDS) == P6['images'] and VAL_VERDICT['verdict'] in ('PASS', 'FALLBACK', 'FAIL')
    assert all(evidence.load_bundle(BUNDLES, u)['uid'] == u for u in P6['uids'])

In [ ]:
# Cell 9: the 20 evidence overlays (validation)
FIG = cp5.overlay_figure(PANELS, OUT / 'cp5_overlays_val.png')
from IPython.display import Image as _Image, display
display(_Image(filename=str(FIG)))
print('saved', FIG, '|', len(PANELS), 'images')

In [ ]:
# Cell 10: compute record, then release the runtime (last cell; everything above is saved on Drive)
COMPUTE = session.finish(OUT, T0, CU_PER_HOUR, name='compute_cp5.json',
                         extra={'git_commit': info['git_commit'], 'gpu': info.get('gpu_name'), 'est_cu': EST_CU,
                                'cu_left_before': CU_LEFT})
RELEASED = session.release(DRY)
if DRY:
    assert RELEASED == 'dry', RELEASED